# Stage 00a — PatSeer Download

Downloads patent drawing images from PatSeer via Selenium and saves them into
`raw/{patent_id}/` using the canonical filename convention:

```
US20220267016A1_img003.png   ← individual figure crop
US20220267016A1_D00005.png   ← full drawing sheet
US20220267016A1_FAT001.png   ← composite sheet
```

A `{patent_id}_download_manifest.json` is saved in each patent folder.

**Run cells top-to-bottom.** Cell 2 opens a Chrome window — log in to PatSeer
when prompted, then press Enter in the output area to start the download loop.

---

| Cell | What it does |
|------|--------------|
| 1 | Load config, set run parameters |
| 2 | Launch PatSeer downloader |
| 3 | Print download coverage report |

In [ ]:
import sys
from pathlib import Path

repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.config_loader import load_config

cfg = load_config()

# ─── Run parameters (edit here) ───────────────────────────────────────────────
OUTPUT_DIR  = cfg["paths"]["raw_images"]   # override with Path("...") if needed
START_FROM  = 1      # record index to start from (set higher to resume)
END_AT      = None   # last record to download; None = use TOTAL_RECORDS from script
# ──────────────────────────────────────────────────────────────────────────────

print(f"Output dir  : {OUTPUT_DIR}")
print(f"Start from  : {START_FROM}")
print(f"End at      : {END_AT or 'all records'}")

In [ ]:
import src.patseer_downloader as dl

# Patch module-level configuration before launching
dl.OUTPUT_DIR = OUTPUT_DIR
dl.START_FROM = START_FROM
if END_AT is not None:
    dl.TOTAL_RECORDS = END_AT

# A Chrome window will open — log in to PatSeer, then press Enter here.
# clean_filename() is applied to every saved file inside download_images().
dl.main()

In [ ]:
import re

raw_dir = OUTPUT_DIR

patents_attempted  = 0
patents_with_img   = 0
patents_with_d     = 0
patents_with_fat   = 0
total_img          = 0
total_d            = 0
total_fat          = 0

for patent_dir in sorted(raw_dir.iterdir()):
    if not patent_dir.is_dir():
        continue
    patents_attempted += 1

    imgs  = [p for p in patent_dir.glob("*.png") if "_img" in p.name.lower()]
    dfiles = [p for p in patent_dir.glob("*.png") if re.search(r"_d\d{4,}", p.name, re.IGNORECASE)]
    fats  = [p for p in patent_dir.glob("*.png") if "_fat" in p.name.lower()]

    if imgs:   patents_with_img += 1
    if dfiles: patents_with_d   += 1
    if fats:   patents_with_fat += 1

    total_img += len(imgs)
    total_d   += len(dfiles)
    total_fat += len(fats)

def _pct(n):
    return f"{n / patents_attempted * 100:.0f}%" if patents_attempted else "N/A"

avg_imgs = total_img / patents_with_img if patents_with_img else 0

print("══════════════════════════════════════")
print("Download Coverage Report")
print("══════════════════════════════════════")
print(f"Patents attempted      : {patents_attempted}")
print(f"Patents with img files : {patents_with_img}  ({_pct(patents_with_img)})")
print(f"Patents with D files   : {patents_with_d}  ({_pct(patents_with_d)})")
print(f"Patents with FAT files : {patents_with_fat}  ({_pct(patents_with_fat)})")
print(f"Total img files        : {total_img}")
print(f"Total D files          : {total_d}")
print(f"Total FAT files        : {total_fat}")
print(f"Average imgs/patent    : {avg_imgs:.1f}")
print("══════════════════════════════════════")